# 文本分类实例

## Step1 导入相关包

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset

## Step2 加载数据集

In [ ]:
dataset = load_dataset("csv", data_files="./ChnSentiCorp_htl_all.csv", split="train")
dataset = dataset.filter(lambda x: x["review"] is not None)
dataset

## Step3 划分数据集

In [ ]:
datasets = dataset.train_test_split(test_size=0.1)
datasets

## Step4 数据集预处理

In [ ]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")


def process_function(examples):
    tokenized_examples = tokenizer(examples["review"], max_length=128, truncation=True)
    tokenized_examples["labels"] = examples["label"]
    return tokenized_examples


tokenized_datasets = datasets.map(
    process_function, batched=True, remove_columns=datasets["train"].column_names
)
tokenized_datasets

## Step5 创建模型

In [ ]:
def model_init():
    model = AutoModelForSequenceClassification.from_pretrained("hfl/rbt3")
    return model

## Step6 创建评估函数

In [ ]:
!pip install evaluate

In [ ]:
import evaluate

acc_metric = evaluate.load("accuracy")
f1_metirc = evaluate.load("f1")

In [ ]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict
    predictions = predictions.argmax(axis=-1)
    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metirc.compute(predictions=predictions, references=labels)
    acc.update(f1)
    return acc

## Step7 创建 TrainingArguments

In [ ]:
train_args = TrainingArguments(
    # 输出文件夹
    output_dir="./checkpoints",
    # 训练时的 batch_size
    per_device_train_batch_size=64,
    # 验证时的 batch_size
    per_device_eval_batch_size=128,
    # log 打印的频率
    logging_steps=50,
    # 评估策略
    eval_strategy="epoch",
    # 保存策略
    save_strategy="epoch",
    # 最大保存数
    save_total_limit=3,
    # 学习率
    learning_rate=2e-5,
    # weight_decay
    weight_decay=0.01,
    # 设定评估指标
    metric_for_best_model="eval_loss",
    # 训练完成后加载最优模型
    load_best_model_at_end=True,
    report_to="none",
)

## Step8 创建 Trainer

In [ ]:
from transformers import DataCollatorWithPadding

trainer = Trainer(
    model_init=model_init,
    args=train_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=eval_metric,
)

## Step9 模型训练

In [ ]:
trainer.train()

## Step9 模型训练(自动搜索)

In [ ]:
!pip install optuna

In [ ]:
!pip freeze > requirements.txt

In [ ]:
trainer.hyperparameter_search(
    compute_objective=lambda x: x["eval_f1"],
    direction="maximize",
    n_trials=10,
    backend="optuna",
)

In [ ]:
def default_hp_space_optuna(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-4, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 1, 5),
        "seed": trial.suggest_int("seed", 1, 40),
        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size", [4, 8, 16, 32, 64]
        ),
        "optim": trial.suggest_categorical("optim", ["sgd", "adamw_torch"]),
    }


trainer.hyperparameter_search(
    hp_space=default_hp_space_optuna,
    compute_objective=lambda x: x["eval_f1"],
    direction="maximize",
    n_trials=10,
    backend="optuna",
)